# EDA de Calidad e Integridad - Eje Logistico

Notebook de analisis exploratorio para `dataset_logistico.csv`, enfocado en calidad de datos, distribuciones, normalidad, relaciones entre variables y validacion de la logica operativa del abastecimiento.

## 1. Librerias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import shapiro

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 2. Carga del dataset

In [ ]:
RUTA_DATASET = "../data/dataset_logistico.csv"

df = pd.read_csv(RUTA_DATASET)
df["dt"] = pd.to_datetime(df["dt"], errors="coerce")

print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")
display(df.head())

## 3. Analisis de calidad e integridad de los datos

In [ ]:
df.info()

print("\nValores nulos por columna:")
display(df.isna().sum().sort_values(ascending=False).to_frame("nulos"))

print(f"\nFilas duplicadas: {df.duplicated().sum():,}")

print("\nResumen estadistico:")
display(df.describe(include="all").T)

## 4. Auditoria de integridad: valores nulos y duplicados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(df.isna(), cbar=False, ax=axes[0])
axes[0].set_title("Mapa de valores faltantes")
axes[0].set_xlabel("Columnas")
axes[0].set_ylabel("Registros")

duplicados = pd.DataFrame({
    "estado": ["No duplicadas", "Duplicadas"],
    "cantidad": [len(df) - df.duplicated().sum(), df.duplicated().sum()]
})
sns.barplot(data=duplicados, x="estado", y="cantidad", ax=axes[1], color="#378ADD")
axes[1].set_title("Auditoria de duplicados")
axes[1].set_xlabel("Estado")
axes[1].set_ylabel("Cantidad")

plt.tight_layout()
plt.show()

## 5. Composicion del inventario: cantidad de productos por categoria

In [ ]:
col_categoria = "nombre_categoria_n1" if "nombre_categoria_n1" in df.columns else None

if col_categoria is not None:
    conteo_categoria = (
        df.groupby(col_categoria, observed=True)["product_id"]
        .nunique()
        .sort_values(ascending=False)
        .reset_index(name="productos_unicos")
    )

    plt.figure(figsize=(12, 6))
    sns.barplot(data=conteo_categoria, y=col_categoria, x="productos_unicos", color="#185FA5")
    plt.title("Cantidad de productos por categoria")
    plt.xlabel("Productos unicos")
    plt.ylabel("Categoria")
    plt.tight_layout()
    plt.show()

    display(conteo_categoria.head(20))

## 6. Analisis de distribuciones y normalidad

In [ ]:
columnas_numericas = [
    "sale_amount",
    "discount",
    "horas_con_stock",
    "venta_lag_1",
    "venta_lag_7",
    "venta_promedio_7d",
    "venta_promedio_14d",
    "stock_lag_1",
]

columnas_numericas = [col for col in columnas_numericas if col in df.columns]
columnas_numericas

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(columnas_numericas):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color="#185FA5")
    axes[i].set_title(f"Distribucion de {col}")

for j in range(i + 1, len(axes)):
    axes[j].set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
resultados_normalidad = []

for col in columnas_numericas:
    serie = df[col].dropna()
    muestra = serie.sample(min(len(serie), 5000), random_state=42) if len(serie) > 5000 else serie
    estadistico, p_valor = shapiro(muestra)
    resultados_normalidad.append({
        "variable": col,
        "estadistico_shapiro": estadistico,
        "p_valor": p_valor,
        "normal": "Si" if p_valor >= 0.05 else "No"
    })

df_normalidad = pd.DataFrame(resultados_normalidad).sort_values("p_valor")
display(df_normalidad)

## 7. Deteccion de valores atipicos (boxplots)

In [ ]:
plt.figure(figsize=(16, 6))
sns.boxplot(data=df[columnas_numericas], orient="h", palette="Blues")
plt.title("Boxplots de variables numericas")
plt.xlabel("Valor")
plt.tight_layout()
plt.show()

## 8. Diagnostico temporal del suministro

In [ ]:
ventas_diarias = df.groupby("dt", as_index=False)["sale_amount"].sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.lineplot(data=ventas_diarias, x="dt", y="sale_amount", ax=axes[0], color="#185FA5")
axes[0].set_title("Demanda total diaria")
axes[0].set_xlabel("Fecha")
axes[0].set_ylabel("Demanda total")
axes[0].tick_params(axis="x", rotation=45)

demanda_dia_semana = df.groupby("dia_semana", as_index=False)["sale_amount"].mean()
sns.barplot(data=demanda_dia_semana, x="dia_semana", y="sale_amount", ax=axes[1], color="#378ADD")
axes[1].set_title("Demanda promedio por dia de la semana")
axes[1].set_xlabel("Dia de la semana")
axes[1].set_ylabel("Demanda promedio")

plt.tight_layout()
plt.show()

## 9. Productos con mayor demanda por tipo

In [ ]:
df["tipo_producto_app"] = (
    df["tipo_producto_app"]
    .astype(str)
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("ascii")
)

top_productos_clinicos = (
    df[df["tipo_producto_app"].str.contains("Clinico", na=False)]
    .groupby(["product_id", "name_products"], observed=True)["sale_amount"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)

top_productos_alimentos = (
    df[df["tipo_producto_app"].str.contains("Alimento", na=False)]
    .groupby(["product_id", "name_products"], observed=True)["sale_amount"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

if not top_productos_clinicos.empty:
    sns.barplot(data=top_productos_clinicos, y="name_products", x="sale_amount", color="#185FA5", ax=axes[0])
    axes[0].set_ylabel("Producto clinico")
else:
    axes[0].text(0.5, 0.5, "No hay productos clinicos clasificados", ha="center", va="center")
    axes[0].set_axis_off()
axes[0].set_title("Top 15 productos clinicos por demanda total")
axes[0].set_xlabel("Demanda total")

if not top_productos_alimentos.empty:
    sns.barplot(data=top_productos_alimentos, y="name_products", x="sale_amount", color="#378ADD", ax=axes[1])
    axes[1].set_ylabel("Alimento")
else:
    axes[1].text(0.5, 0.5, "No hay alimentos clasificados", ha="center", va="center")
    axes[1].set_axis_off()
axes[1].set_title("Top 15 alimentos por demanda total")
axes[1].set_xlabel("Demanda total")

plt.tight_layout()
plt.show()

display(top_productos_clinicos)
display(top_productos_alimentos)

## 10. Analisis de relaciones, covarianza y dispersion

In [ ]:
corr = df[columnas_numericas].corr(numeric_only=True)
cov = df[columnas_numericas].cov(numeric_only=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[0])
axes[0].set_title("Mapa de calor de correlacion - Eje logistico")

sns.heatmap(cov, annot=True, fmt=".2f", cmap="YlGnBu", ax=axes[1])
axes[1].set_title("Matriz de covarianza - Variables logisticas")

plt.tight_layout()
plt.show()

In [ ]:
variables_predictoras = [
    "stock_lag_1",
    "venta_lag_1",
    "venta_lag_7",
    "venta_promedio_7d",
    "venta_promedio_14d",
    "discount",
]

variables_predictoras = [col for col in variables_predictoras if col in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(variables_predictoras):
    sns.scatterplot(data=df.sample(min(5000, len(df)), random_state=42), x=col, y="sale_amount", alpha=0.3, ax=axes[i])
    axes[i].set_title(f"{col} vs sale_amount")

for j in range(i + 1, len(axes)):
    axes[j].set_axis_off()

plt.tight_layout()
plt.show()

## 11. Stock, descuentos y eventos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df.sample(min(5000, len(df)), random_state=42), x="stock_lag_1", y="sale_amount", alpha=0.25, ax=axes[0])
axes[0].set_title("Demanda vs stock del dia anterior")
axes[0].set_xlabel("stock_lag_1")
axes[0].set_ylabel("sale_amount")

sns.scatterplot(data=df.sample(min(5000, len(df)), random_state=42), x="discount", y="sale_amount", alpha=0.25, ax=axes[1])
axes[1].set_title("Demanda vs descuento")
axes[1].set_xlabel("discount")
axes[1].set_ylabel("sale_amount")

plt.tight_layout()
plt.show()

for col in ["holiday_flag", "activity_flag"]:
    if col in df.columns:
        display(df.groupby(col, observed=True)["sale_amount"].agg(["count", "mean", "median", "sum"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=df.groupby("holiday_flag", as_index=False)["sale_amount"].mean(), x="holiday_flag", y="sale_amount", ax=axes[0], color="#378ADD")
axes[0].set_title("Demanda promedio segun holiday_flag")

sns.barplot(data=df.groupby("activity_flag", as_index=False)["sale_amount"].mean(), x="activity_flag", y="sale_amount", ax=axes[1], color="#185FA5")
axes[1].set_title("Demanda promedio segun activity_flag")

plt.tight_layout()
plt.show()

if "horas_con_stock" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df.sample(min(5000, len(df)), random_state=42), x="horas_con_stock", y="sale_amount", alpha=0.3)
    plt.title("Demanda vs horas con stock disponible")
    plt.xlabel("Horas con stock")
    plt.ylabel("Demanda")
    plt.tight_layout()
    plt.show()

## 12. Conclusiones del EDA

In [ ]:
print("Resumen del EDA logistico:")
print("- Se audito la integridad del dataset mediante nulos, duplicados y revision estructural.")
print("- Se mantuvieron los analisis de distribucion, normalidad, covarianza, correlacion y dispersion con variables del nuevo dataset.")
print("- Se reemplazo el antiguo diagnostico de fechas por un analisis temporal de la demanda.")
print("- Se anadieron rankings separados para productos clinicos y alimentos.")
print("- El notebook queda listo para sustentar el texto del proyecto con el nuevo eje logistico.")